# Итоговый проект по продуктовой аналитике маркетплейса Olist

Цель — изучить возвращаемость покупателей, оценить признаки product/market fit, определить продуктовые метрики и выбрать гипотезу для будущего A/B-теста.

Работа последовательно отвечает на шесть задач проекта. Данные: `olist_customers_dataset.csv`, `olist_orders_dataset.csv`, `olist_order_items_dataset.csv`. **Выгрузка содержит исторические заказы, а не результаты A/B-теста.** Поэтому ниже выполняются анализ и проектирование проверки; эффект гипотез не выдаётся за экспериментально доказанный.

## Как запустить
Положить три исходных CSV в папку `data/private`, затем выполнить все ячейки сверху вниз. Нужны Python, pandas, numpy, matplotlib и seaborn. Версия для GitHub очищена от выводов ячеек. Выполненная копия находится в отдельном приватном архиве.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13})

FILENAMES = ['olist_customers_dataset.csv', 'olist_orders_dataset.csv',
             'olist_order_items_dataset.csv']
candidates = [Path('data/private')]
DATA_DIR = next((p for p in candidates if all((p / f).is_file()
                for f in FILENAMES)), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        'Положите три CSV в data/private и запускайте ноутбук из корня проекта.')
customers = pd.read_csv(
    DATA_DIR / FILENAMES[0],
    dtype={
        'customer_zip_code_prefix': 'string'})
orders = pd.read_csv(DATA_DIR / FILENAMES[1])
items = pd.read_csv(DATA_DIR / FILENAMES[2])
for col in orders.columns:
    if col.endswith('_date') or col.endswith(
            '_at') or col.endswith('_timestamp'):
        orders[col] = pd.to_datetime(orders[col], errors='raise')
items['shipping_limit_date'] = pd.to_datetime(
    items['shipping_limit_date'], errors='raise')


## Проверка качества и объединение данных

`customer_id` связывает заказ с записью клиента, а `customer_unique_id` идентифицирует одного человека между заказами. Сначала соединяем заказы с покупателями, затем агрегируем стоимость позиций до уровня заказа. Прямое соединение позиций с заказами размножило бы строки и исказило количество заказов.

Пропуски в датах доставки для отменённых и ещё не доставленных заказов ожидаемы. Для задержек используем только строки с обеими датами доставки. Факт доставки определяем по статусу, поэтому отдельный пропуск даты сам по себе не исключает заказ из retention.

In [ ]:
audit = pd.DataFrame([
    {'Таблица': name, 'Строки': len(df), 'Полные дубли': df.duplicated().sum(),
     'Ячейки с пропусками': df.isna().sum().sum()}
    for name, df in [
        ('customers', customers), ('orders', orders), ('items', items)
    ]
])
display(audit)
display(orders.isna().sum().rename('Пропуски').to_frame())
display(orders['order_status'].value_counts().rename('Заказы').to_frame())
assert customers['customer_id'].is_unique
assert orders['order_id'].is_unique
assert not items.duplicated(['order_id', 'order_item_id']).any()
assert not customers[['customer_id', 'customer_unique_id']].isna().any().any()
assert not orders[['order_id', 'customer_id',
                   'order_purchase_timestamp']].isna().any().any()
assert (items[['price', 'freight_value']] >= 0).all().all()
assert set(items['order_id']).issubset(set(orders['order_id']))

order_values = items.groupby('order_id', as_index=False).agg(
    order_value=('price', 'sum'), freight_value=('freight_value', 'sum'),
    item_count=('order_item_id', 'size'))
base = orders.merge(
    customers,
    on='customer_id',
    how='left',
    validate='one_to_one')
assert base['customer_unique_id'].notna().all()
base = base.merge(
    order_values,
    on='order_id',
    how='left',
    validate='one_to_one')
assert len(base) == len(orders)
base['order_month'] = base['order_purchase_timestamp'].dt.to_period('M')
buy = base.loc[base['order_status'].eq('delivered')].copy()
assert buy['order_value'].notna().all()
print('Период создания заказов:', base['order_purchase_timestamp'].min(),
      '—', base['order_purchase_timestamp'].max())
print('Последний созданный доставленный заказ:',
      buy['order_purchase_timestamp'].max())
print('Доставленные заказы без даты доставки:',
      buy['order_delivered_customer_date'].isna().sum())
print('Доставленные заказы без даты подтверждения оплаты:',
      buy['order_approved_at'].isna().sum())
display(base.groupby('order_month', observed=True).agg(
    all_orders=('order_id', 'nunique'),
    delivered=('order_status', lambda s: s.eq('delivered').sum())))


## Задача 1 Месячный retention

**Определение заказа.** В основной версии учитываем `delivered`: это состоявшаяся покупка, по которой покупатель получил товар. Статусы `canceled` и `unavailable` не считаем успешной покупкой; незавершённые статусы также не включаем. Месяц события определяем по **дате оформления** `order_purchase_timestamp`, а не дате доставки. Такой подход измеряет возвращение к оформлению успешно завершившегося заказа. Выбор сделан явно: для retention любых попыток оформления понадобилось бы иное определение события.

Когорта — календарный месяц первого доставленного заказа покупателя. Возраст `M0` — месяц первой покупки, `M1` — следующий календарный месяц, `M3` — третий месяц после месяца первой покупки. Это не интервалы по 30 дней.

**Retention Mᵏ = число уникальных покупателей когорты с заказом в Mᵏ / исходный размер когорты.** Возвращение не обязано быть непрерывным. Покупатель, пропустивший M1 и вернувшийся в M3, учитывается в M3.

Последний месяц массовых покупок — август 2018: последняя дата создания доставленного заказа 29 августа. В сентябре и октябре есть только единичные незавершённые или отменённые заказы, поэтому не используем эти месяцы как месяцы нулевого retention. Основная учебная оценка строится до августа 2018 включительно; отдельно проверяем результат на полностью завершённых месяцах до июля. Возможна также неполная зрелость статусов последних заказов.

Нулевое число возвратов в наблюдаемом периоде — **0**, будущий период — **NaN**. Ранние малые когорты сохраняем, поскольку задание просит медиану всех когорт, но отдельно показываем чувствительность к ним.

In [ ]:
LAST_OBSERVED_MONTH = pd.Period('2018-08', freq='M')
LAST_COMPLETE_MONTH = pd.Period('2018-07', freq='M')
buy['cohort_month'] = buy.groupby('customer_unique_id')[
    'order_month'].transform('min')
buy['cohort_age'] = buy['order_month'].astype('int64').sub(
    buy['cohort_month'].astype('int64')
)


def cohort_tables(data, last_month):
    x = data.loc[data['order_month'] <= last_month]
    counts = x.groupby(['cohort_month', 'cohort_age'], observed=True)[
        'customer_unique_id'].nunique().unstack(fill_value=0)
    counts = counts.reindex(columns=range(
        int(counts.columns.max()) + 1), fill_value=0)
    counts = counts.astype(float)
    for cohort in counts.index:
        for age in counts.columns:
            if cohort + age > last_month:
                counts.loc[cohort, age] = np.nan
    sizes = counts[0].astype(int)
    retention = counts.div(sizes, axis=0)
    return counts, sizes, retention


cohort_counts, cohort_sizes, retention = cohort_tables(
    buy, LAST_OBSERVED_MONTH)
_, complete_sizes, complete_retention = cohort_tables(buy, LAST_COMPLETE_MONTH)
median_m1 = retention[1].dropna().median()
best_m3 = retention[3].idxmax()
best_m3_value = retention.loc[best_m3, 3]

answer_1 = pd.DataFrame({
    'Размер когорты': cohort_sizes,
    'M1 %': retention[1] * 100,
    'M3 %': retention[3] * 100
})
display(answer_1)
print(f'Медианный M1 всех зрелых когорт: {median_m1:.4%}')
print(f'Максимальный M3: {best_m3}, {best_m3_value:.4%}')
print('Вернувшиеся в M3 / размер когорты:',
      int(cohort_counts.loc[best_m3, 3]), '/', cohort_sizes.loc[best_m3])
print(f'Медианный M1 до июля 2018: {complete_retention[1].median():.4%}')
print(
    f'Медианный M1 регулярных когорт с января 2017: '
    f'{retention.loc["2017-01":, 1].median():.4%}')
assert retention[0].eq(1).all()
assert retention.stack().between(0, 1).all()


In [ ]:
# M0 = 100% не выводим: он скрывает различия в последующих месяцах.
plot_retention = retention.loc['2017-01':].drop(columns=0) * 100
fig, ax = plt.subplots(figsize=(15, 8))
sns.heatmap(plot_retention, mask=plot_retention.isna(), cmap='YlGnBu',
            annot=True, fmt='.2f', vmin=0, vmax=1,
            cbar_kws={'label': 'Retention, %'}, ax=ax, annot_kws={'size': 8})
ax.set(title='Месячный retention по доставленным заказам',
       xlabel='Месяцы после первой покупки', ylabel='Когорта первой покупки')
ax.set_yticklabels([str(x) for x in plot_retention.index], rotation=0)
plt.tight_layout()
plt.show()


### Ответ по первой задаче

Медианный retention первого месяца по всем когортам с наблюдаемым M1 — **0,4788%**. Это медиана долей по когортам, без взвешивания на размер. Максимальный retention M3 у когорты **июня 2017 года: 0,4281%**, или **13 из 3037 покупателей**.

Исключение неполного августа из окна наблюдения даёт медиану M1 **0,4636%**. Для регулярных когорт с января 2017 при основном окне — **0,4939%**. Вывод о низкой возвращаемости устойчив к этим решениям. Когорта декабря 2016 состоит из одного покупателя, вернувшегося в январе: её 100% нельзя интерпретировать как массовый успех продукта.

На тепловой карте отсутствующие будущие месяцы оставлены пустыми. Ранние малые когорты исключены только из визуализации, но присутствуют в таблице и обязательном расчёте медианы.

## Задача 2 Оценка product market fit

Низкий месячный retention и низкий уровень последующих повторных покупок **не дают убедительных признаков PMF для модели регулярных покупок**. Однако только по retention нельзя окончательно доказать отсутствие PMF: товары могут приобретаться редко, а в данных нет категорий, затрат на привлечение и опросов клиентов. Оценку формулируем как «PMF пока не подтверждён», а не как доказанную убыточность.

Для сравнения когорт строим кривые за одинаковые первые шесть месяцев. Средневзвешенную кривую рассчитываем по одному фиксированному набору зрелых когорт: иначе на каждом возрасте менялся бы состав покупателей, а позднее улучшение могло бы оказаться артефактом.

In [ ]:
regular_cohorts = retention.index >= pd.Period('2017-01')
mature_cohorts = retention.index + 6 <= LAST_COMPLETE_MONTH
eligible = retention.index[regular_cohorts & mature_cohorts]
weights = cohort_sizes.loc[eligible]
weighted_curve = retention.loc[eligible, range(1, 7)].mul(
    weights, axis=0).sum() / weights.sum()
fig, ax = plt.subplots(figsize=(10, 5))
for cohort in eligible:
    ax.plot(range(1, 7), retention.loc[cohort, range(1, 7)] * 100,
            color='#8899aa', alpha=.25, linewidth=1)
ax.plot(range(1, 7), weighted_curve * 100, color='#146c94', marker='o',
        linewidth=3, label='Взвешенное среднее фиксированных зрелых когорт')
ax.set(
    title='После первой покупки возвращается небольшая доля покупателей',
    xlabel='Месяцы после первой покупки',
    ylabel='Retention, %',
    xticks=range(
        1,
        7),
    ylim=(
        0,
        1))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

buyer_order_counts = buy.groupby('customer_unique_id')['order_id'].nunique()
repeat_share = buyer_order_counts.gt(1).mean()
print(
    f'Покупателей с двумя и более доставленными заказами за весь период: {repeat_share:.2%}')
display((weighted_curve * 100).rename('Взвешенный retention %').to_frame())


### Вывод и фокус продукта

За весь наблюдаемый период только **3,00%** покупателей сделали более одного доставленного заказа. Показатель диагностический: сроки наблюдения разных покупателей различаются, поэтому это не равнозначная замена когортному retention.

Перед масштабированием следует разобраться с повторной ценностью продукта и качеством исполнения заказов. Возможные причины низкого retention: редкая потребность в товарах, отсутствие повода для повторного заказа, недостаточный ассортимент, неудобство оплаты, отмены и нарушения сроков. Это **гипотезы причин**, а не установленные причинные связи.

Рекомендуется исследовать причины отказа и повторных покупок через интервью и данные воронки, анализировать удержание по типам товаров при появлении категорий, проверить экономику привлечения и маржинальность. Параллельно устранить измеримые проблемы исполнения заказов. Рост аудитории сам по себе не подтверждает рост прибыли.

## Задача 3 Пять основных метрик

| № | Метрика | Определение | Связь с бизнесом |
|---|---|---|---|
| 1 | GMV доставленных заказов за месяц | Сумма `price` всех позиций доставленных заказов, оформленных в месяце | Объём продаж товаров |
| 2 | Monthly Paying Users, MPU | Уникальные `customer_unique_id` с доставленным заказом, оформленным в месяце | Аудитория, получившая ценность |
| 3 | Конверсия нового посетителя в первую покупку | Новые посетители с первой успешной покупкой в течение 30 дней / новые посетители с полным 30-дневным наблюдением | Заинтересованность и активация новых клиентов |
| 4 | Месячный retention M1 | Покупатели когорты с повторной покупкой в следующем календарном месяце / размер когорты | Продолжение использования продукта |
| 5 | GMV на покупателя за месяц | GMV месяца / MPU месяца | Денежное выражение покупательской активности |

Третью метрику нельзя корректно посчитать: нужны идентификаторы посетителей, события первого визита и связь посетителя с покупателем. Все клиенты текущей таблицы связаны с заказами, поэтому использовать их как всех посетителей нельзя.

GMV — **оборот товаров, а не выручка или прибыль маркетплейса**. `freight_value` не включаем в стоимость товаров. GMV на покупателя — аналог ARPPU по обороту, но не настоящий ARPPU по доходу платформы. Для расчёта прибыли нужны комиссия, себестоимость услуг, возвраты, операционные и маркетинговые расходы. Четыре доступные метрики — измеримые ориентиры, а не достаточное условие максимизации прибыли.

In [ ]:
monthly = buy.groupby('order_month', observed=True).agg(
    gmv=('order_value', 'sum'), mpu=('customer_unique_id', 'nunique'),
    orders=('order_id', 'nunique'))
monthly['gmv_per_buyer'] = monthly['gmv'] / monthly['mpu']
monthly['m1_retention_pct'] = retention[1] * 100
monthly['new_buyers'] = cohort_sizes
monthly['returning_buyers'] = monthly['mpu'] - monthly['new_buyers']
display(monthly)
assert np.isclose(monthly['gmv'].sum(), buy['order_value'].sum())
reconstructed_gmv = monthly['gmv_per_buyer'] * monthly['mpu']
assert np.isclose(reconstructed_gmv.sum(), monthly['gmv'].sum())

regular = monthly.loc['2017-01':'2018-08'].copy()
labels = regular.index.astype(str)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
spec = [('gmv', 'Объём продаж товаров', 'GMV, BRL', 1),
        ('mpu', 'Активные покупатели', 'MPU, покупателей', 1),
        (
            'm1_retention_pct', 'Retention следующего месяца по когортам',
            'M1, %', 100
),
    (
            'gmv_per_buyer', 'Оборот на одного покупателя',
            'BRL на покупателя', 1
)]
for ax, (col, title, ylabel, _) in zip(axes.flat, spec):
    vals = regular[col].copy()
    if col == 'm1_retention_pct':
        # M1 июльской когорты приходится на неполный август.
        complete_vals = vals.copy()
        complete_vals.loc['2018-07':] = np.nan
        ax.plot(range(len(labels)), complete_vals, marker='o', color='#146c94')
        ax.scatter([len(labels) - 2], [vals.iloc[-2]], color='#d27b29',
                   label='M1 включает неполный август', zorder=4)
    else:
        ax.plot(range(len(labels) - 1),
                vals.iloc[:-1],
                marker='o',
                color='#146c94')
        ax.plot([len(labels) - 2, len(labels) - 1],
                vals.iloc[-2:], '--', color='#d27b29')
        ax.scatter([len(labels) - 1], [vals.iloc[-1]],
                   color='#d27b29', label='Неполный август')
    ax.set(title=title, ylabel=ylabel)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=8)
    ax.legend(fontsize=8)
    ax.ticklabel_format(axis='y', style='plain')
plt.tight_layout()
plt.show()

recent = monthly.loc['2018-01':'2018-07']
print('Средние месячные значения за январь–июль 2018:')
display(recent[['gmv', 'mpu', 'gmv_per_buyer']].mean().to_frame('Среднее'))
print(f'Доля новых среди активных покупателей за январь–июль 2018: '
      f'{recent.new_buyers.sum() / recent.mpu.sum():.2%}')


### Интерпретация метрик

В 2017 году оборот и активная аудитория росли; ноябрь выделяется всплеском. Причина всплеска не установлена по этим таблицам. В 2018 году рост не выглядит устойчивым: движение GMV во многом следует за числом покупателей, а M1 остаётся ниже 1% у регулярных когорт.

Денежную активность можно повышать через увеличение частоты покупок или суммы заказа, но оценивать это вместе с маржинальностью и клиентским опытом. Увеличение среднего оборота на покупателя за счёт ухудшения доступности цен может снизить активацию и retention.

Август отмечен отдельно как неполный: снижение или рост относительно июля нельзя трактовать как полный месячный результат без сопоставимого периода. Отсутствующие месяцы первых небольших когорт не заполняются искусственными продажами.

## Задача 4 Приоритизация гипотез через ICE

Используем данные **с 1 июня 2017 года до конца выгрузки**, как указано в задании. Для предполагаемой конверсии в повторный заказ используем медиану M1 из задачи 1: **0,4788%**, то есть `0.00478771`, а не `0.4788`.

**Impact измеряем через ожидаемое дополнительное количество доставленных заказов**, затем переводим количество в балл по шкале задания. `ICE = Impact × Confidence × Ease`; использование среднего произведения вместо произведения изменило бы значения, но не ранжирование.

Модель эффекта:
1. Исправление процессинга: каждый отменённый заказ становится доставленным. Основной сценарий учитывает и `canceled`, и `unavailable`, поскольку оба статуса обозначают отмену и условие обещает убрать все отмены. Это сильное **допущение задания**: на практике исправление процессинга не обязательно устраняет отсутствие товара. Ниже приведён также сценарий только `canceled`.
2. Сокращение времени отгрузки: уникальные покупатели с опоздавшим доставленным заказом × медиана M1. Один покупатель учитывается один раз, даже при нескольких опоздавших заказах. Предполагаем один дополнительный заказ с заданной вероятностью. Это учебная модель потенциального эффекта, а не доказанный прирост относительно контроля.
3. Новый способ оплаты: уникальные покупатели с единственным доставленным заказом за всю историю, купившие с июня 2017 × медиана M1. Так не включаем уже повторных покупателей. Тоже предполагаем один потенциальный дополнительный заказ.

Опоздание определяется сравнением **календарных дат** фактической и обещанной доставки: доставка вечером в обещанный день не считается опозданием. Последние малые месяцы не используются для retention, но их отмены сохраняются в ICE согласно периоду задания; чувствительность к их исключению проверяется отдельно.

In [ ]:
impact_mask = base['order_purchase_timestamp'] >= '2017-06-01'
impact_period = base.loc[impact_mask].copy()
delivered_mask = buy['order_purchase_timestamp'] >= '2017-06-01'
delivered_period = buy.loc[delivered_mask].copy()
known_delivery = delivered_period.dropna(subset=[
    'order_delivered_customer_date', 'order_estimated_delivery_date']).copy()
actual_delivery = (
    known_delivery['order_delivered_customer_date'].dt.normalize()
)
promised_delivery = (
    known_delivery['order_estimated_delivery_date'].dt.normalize()
)
late = actual_delivery > promised_delivery
late_buyers = known_delivery.loc[late, 'customer_unique_id'].nunique()
all_cancellations = impact_period['order_status'].isin(
    ['canceled', 'unavailable']).sum()
only_canceled = impact_period['order_status'].eq('canceled').sum()
buyer_order_counts = buy.groupby('customer_unique_id')['order_id'].nunique()
single_ids = buyer_order_counts.index[buyer_order_counts.eq(1)]
single_mask = delivered_period['customer_unique_id'].isin(single_ids)
single_buyers = delivered_period.loc[
    single_mask, 'customer_unique_id'
].nunique()

# Шкала задана для целых заказов. Округляем математическое ожидание вверх
# только при выборе интервала; исходный ожидаемый эффект сохраняем дробным.
upper_bounds = np.array(
    [50, 150, 350, 750, 1550, 3150, 6350, 12750, 25550, 51150])


def impact_score(expected):
    n = int(np.ceil(expected))
    if n < 0 or n > upper_bounds[-1]:
        raise ValueError('Значение вне шкалы задания')
    return int(np.searchsorted(upper_bounds, n, side='left') + 1)


ice = pd.DataFrame({
    'Гипотеза': [
        'Исправить процессинг', 'Ускорить отгрузку', 'Новый способ оплаты'
    ],
    'База эффекта': [all_cancellations, late_buyers, single_buyers],
    'Ожидаемые дополнительные заказы': [
        float(all_cancellations),
        late_buyers * median_m1,
        single_buyers * median_m1
    ],
    'Confidence': [8, 10, 5], 'Ease': [6, 4, 9]})
ice['Impact'] = ice['Ожидаемые дополнительные заказы'].map(impact_score)
ice['ICE'] = ice['Impact'] * ice['Confidence'] * ice['Ease']
display(ice.sort_values('ICE', ascending=False))
print(
    'Опоздавших заказов:',
    late.sum(),
    '; уникальных покупателей:',
    late_buyers)
print('Неопределённая дата доставки среди доставленных:',
      len(delivered_period) - len(known_delivery))
print(
    'Только canceled:',
    only_canceled,
    '; ICE процессинга в узком сценарии:',
    impact_score(only_canceled) * 8 * 6)

sensitivity = []
for title, r in [('Основная оценка', median_m1),
                 ('Без неполного августа', complete_retention[1].median()),
                 (
                     'Регулярные когорты с 2017 года',
                     retention.loc['2017-01':, 1].median()
)]:
    scores = [impact_score(all_cancellations) * 8 * 6,
              impact_score(late_buyers * r) * 10 * 4,
              impact_score(single_buyers * r) * 5 * 9]
    sensitivity.append({'Сценарий': title,
                        'M1 %': r * 100,
                        'ICE процессинга': scores[0],
                        'ICE отгрузки': scores[1],
                        'ICE оплаты': scores[2]})
display(pd.DataFrame(sensitivity))
tail_mask = impact_period['order_purchase_timestamp'] < '2018-09-01'
without_tail = impact_period.loc[tail_mask]
print('Отмены без единичных сентября/октября:',
      without_tail['order_status'].isin(['canceled', 'unavailable']).sum())


### Выбранная гипотеза

| Гипотеза | Ожидаемые дополнительные заказы | Impact | Confidence | Ease | ICE |
|---|---:|---:|---:|---:|---:|
| Исправить процессинг | 974 | 5 | 8 | 6 | **240** |
| Ускорить отгрузку | около 28,98 | 1 | 10 | 4 | 40 |
| Новый способ оплаты | около 384,33 | 4 | 5 | 9 | 180 |

Выбираем **исправление процессинга заказов**. В сценарии только `canceled` база составляет 499 заказов, Impact = 4, ICE = **192**: победитель не меняется. Исключение единичных сентября и октября даёт 955 отмен, то есть тот же Impact = 5. Изменение медианы retention в рассмотренных сценариях также сохраняет ранжирование.

Цифры 974 и 499 — не гарантированные дополнительные продажи, а верхняя оценка при полном устранении соответствующих отмен. Опоздания не обязательно вызваны временем до отгрузки, а отсутствие повторной покупки — способом оплаты. Эти причинные допущения проверяются перед реализацией и экспериментом. У старых и новых клиентов различаются сроки наблюдения; ICE отражает учебную модель по историческому окну, не прогноз за единый будущий месяц.

## Задача 5 Метрики будущего A B теста

Выбранная гипотеза: **если исправить ошибки процессинга, то доля оформленных заказов, завершившихся доставкой, увеличится, потому что станет меньше технических отмен**.

Перед запуском следует выяснить коды причин отмен. Если ошибки мешают до создания заказа, `order_id` не отражает все попытки и потребуется событие начала оформления. Если исправление обязательно для всех по соображениям надёжности, вместо сохранения неисправного контроля нужен иной план оценки. Ниже — дизайн для безопасного улучшения с допустимым текущим вариантом.

### Целевая метрика
**Конверсия оформленного заказа в доставленный в течение 60 дней:** число заказов, доставленных в течение 60 дней после оформления / число оформленных заказов с полным 60-дневным наблюдением. Целевое направление — рост. Механика непосредственно влияет на завершение заказа, а эта метрика связывает исправление с получением товара.

### Прокси метрика
**Доля заказов, отменённых из-за технической ошибки процессинга в течение первых 7 дней:** число таких заказов / все оформленные заказы с полным 7-дневным наблюдением. Целевое направление — снижение. Метрика раньше показывает, сработало ли исправление. Нужны коды причин: статусы `canceled` и `unavailable` сами по себе не выделяют техническую причину.

### Guardrail метрики
- **Доля заказов с повторным ошибочным списанием за 7 дней:** заказов с подтверждённым повторным списанием / оформленные заказы с полным окном. Не должна расти; критическая платёжная ошибка требует остановки. Нужны платёжные события.
- **Средние переменные затраты на оформленный заказ за 60 дней:** затраты на оплату, обработку, доставку и возвраты / оформленные заказы с полным окном. Не должны ухудшить экономику сверх согласованного порога. Нужны данные затрат.
- **Количество обращений по оплате и заказам на 100 назначенных пользователей за 30 дней:** обращения / все назначенные пользователи с полным окном × 100. Не должно существенно расти. Нужны данные поддержки.

Дополнительно контролируем задержки доставки относительно обещанной даты и конверсию начала оформления в создание заказа. Пределы допустимого ухудшения фиксируем с командой **до** запуска на основании исторических данных и бизнес-рисков, а не выбираем после получения результата.

### Организация проверки

Рандомизация на уровне `customer_unique_id`, одинаковый вариант для всех заказов одного пользователя. Набор групп идёт одновременно; анализ — по первоначальному назначению. Повторные заказы одного клиента зависимы: для целевой метрики на уровне заказов используем кластерный bootstrap по пользователям либо другую оценку с кластеризацией. Обычный тест независимых долей на всех заказах без учёта этой зависимости не подходит.

Размер выборки определяем по исторической целевой конверсии, минимальному полезному эффекту, мощности и числу заказов на клиента. Retention 0,4788% не является базовой конверсией оформления в доставку. Набор участников и полное окно наблюдения планируем заранее. При росте целевой метрики и соблюдении guardrail — поэтапное внедрение с мониторингом. В текущей выгрузке нет групп эксперимента: **p-value и эффект A/B-теста сейчас не рассчитываем**.

## Задача 6 Отчёт для продакт менеджера

### Формализация проблемы

Выручка по условиям кейса не растёт. В данных видна слабая возвращаемость после состоявшейся покупки: медианный M1 — **0,4788%**, доля покупателей с повторным доставленным заказом за всю историю — **3,00%**. Доход платформы напрямую не измерен: таблицы дают оборот товаров. Поэтому стагнацию именно выручки принимаем как вводную кейса, а данные используем для диагностики поведения.

### Общие выводы исследования

1. Retention регулярных когорт остаётся низким; лучшая когорта M3 — июнь 2017, **0,4281%**. Нулевые возвраты отделены от ненаблюдаемых будущих периодов, а малые ранние когорты и неполный август учтены в чувствительности.
2. PMF для модели регулярных повторных покупок **пока не подтверждён**. Низкий retention не доказывает отсутствие спроса на товары редкого потребления и не доказывает убытки.
3. Предлагаемые ориентиры — GMV, MPU, конверсия новых посетителей в первую покупку, M1 retention и GMV на покупателя. Для конверсии нужны события посещений, а для прибыли — комиссия и затраты. Все четыре доступные метрики визуализированы с месячной гранулярностью.
4. По заданной модели ICE приоритет — исправление процессинга: **240** против **180** у нового способа оплаты и **40** у ускорения отгрузки. При учёте только статуса `canceled` результат — **192**, и приоритет сохраняется.
5. Будущий эксперимент должен проверять рост завершения заказов, снижение технических отмен и отсутствие вреда оплате, экономике и поддержке. Исторические расчёты Impact не являются доказательством эффекта изменения.

### Рекомендации по продукту

Сначала разобрать причины отмен и подтвердить, какая доля связана с процессингом. Внедрить измерение попыток оформления, ошибок оплаты и причины отмен. Подготовить безопасную проверку исправления с целевой, прокси и защитными метриками; после успешной оценки расширять внедрение постепенно.

Параллельно исследовать повторную потребность клиентов, причины возврата и отказа, качество доставки и ассортимент. Масштабирование привлечения и выход на новые рынки рассматривать после проверки экономики и повторной ценности. При появлении данных о комиссиях, затратах и источниках привлечения дополнить анализ маржинальной прибылью и окупаемостью привлечения.

**Итоговое решение:** приоритет — исправление процессинга с проверкой его реального влияния. Проект завершает этап исследования и подготовки эксперимента; результатов проведённого A/B-теста здесь нет.